# Notebook 00 — Setup and your first local AI call

**Time:** about 30 minutes (plus the one-time model download)

**What you will learn**
- How to start Ollama and download a free model on Arch Linux
- How to send a question to the model from Python
- What is inside a model's reply (text, tokens, time)
- How a *system message* changes the model's behaviour

**Everything runs on your own computer.** No API key, no account, no internet after the model is downloaded.

## Part 1 — One-time setup on Arch Linux

**Easiest:** in a terminal, inside this folder, run `./setup_arch.sh` once. Later, run `./start_workshop.sh`.

Or type the commands yourself, one at a time:

```bash
sudo pacman -S ollama python python-pip       # install Ollama and Python
sudo systemctl enable --now ollama            # start Ollama (and start it at every boot)
ollama pull llama3.2                          # download the model (about 2 GB, needs internet once)
ollama list                                   # llama3.2 should appear in the list
```

Then set up Python for the workshop (inside the workshop folder):

```bash
python -m venv .venv                          # a private Python environment for this workshop
source .venv/bin/activate
python -m pip install -r requirements.txt
jupyter lab                                   # opens JupyterLab in your browser
```

💡 Every time you come back later, you only need: `source .venv/bin/activate` and `jupyter lab` (or `./start_workshop.sh`).

## Part 2 — Check that everything works

In a notebook, click a code cell and press **Shift + Enter** to run it. Always run cells from top to bottom.

In [3]:
# %pip install ollama
# STEP 0 — Is Ollama running? Is the model downloaded?
import ollama

MODEL = "llama3.2"     # local model (bigger one: "qwen2.5:7b")

try:
    installed = [m.model for m in ollama.list().models]
    print("Models on this computer:", installed)
    if MODEL in installed or MODEL + ":latest" in installed:
        print("✅ Ready to go with", MODEL)
    else:
        print(f"⚠️ {MODEL} is missing. Run: ollama pull {MODEL}")
except ConnectionError:
    print("⚠️ Ollama is not running. Start it, then run this again:")
    print("   Arch Linux: sudo systemctl start ollama")
    print("   Windows 11: open Ollama from the Start menu")

Models on this computer: ['llama3.2:latest']
✅ Ready to go with llama3.2


In [4]:
# STEP 1 — A helper: send a message, get the answer back as text
DEFAULT_ROLE = "You are a helpful assistant. Keep answers short."

def ask(prompt, system=DEFAULT_ROLE):
    reply = ollama.chat(
        model=MODEL,
        messages=[{"role": "system", "content": system},
                  {"role": "user", "content": prompt}],
        options={"temperature": 0},    # 0 = more predictable
    )
    return reply.message.content

print("ask() is ready.")

ask() is ready.


## Part 3 — Your first question

`ask()` sends your text to the model and returns its answer. The first call can take 10–60 seconds,
because Ollama loads the model into memory. Later calls are faster.

In [5]:
answer = ask("What is an AI agent? Answer in two sentences.")
print(answer)

An AI agent is a software program that uses artificial intelligence (AI) and machine learning (ML) to make decisions, take actions, and interact with its environment. It can be thought of as a virtual entity that perceives its surroundings, learns from data, and adapts to changing situations to achieve its goals.


## Part 4 — What is inside a reply?

`ask()` only gives back the text. Here we call `ollama.chat()` directly to see the whole reply.

- **Tokens** are small pieces of words. The model reads *input tokens* and writes *output tokens*.
- More tokens = more work for your computer = more time.

In [6]:
message = {"role": "user", "content": "Say hello in five words."}
reply = ollama.chat(model=MODEL, messages=[message])
seconds = reply.total_duration / 1e9     # nanoseconds → seconds

print("Answer        :", reply.message.content)
print("Input tokens  :", reply.prompt_eval_count)
print("Output tokens :", reply.eval_count)
print("Total time    :", round(seconds, 1), "seconds")

Answer        : Hello there, it's nice to meet you.
Input tokens  : 31
Output tokens : 11
Total time    : 1.3 seconds


## Part 5 — The system message sets the model's role

The **system message** tells the model *how* to behave. The **user message** is the actual question.
Same question, three different system messages:

In [7]:
question = "What is an AI agent?"

roles = [
    "Explain to a 10-year-old child in one short sentence.",
    "You are a poet. Answer with a 3-line poem.",
    "You are an engineer. Answer in one technical sentence.",
]

for role in roles:
    print("ROLE:", role)
    print("ANSWER:", ask(question, system=role))
    print("-" * 60)

ROLE: Explain to a 10-year-old child in one short sentence.
ANSWER: An AI agent is like a super smart computer that can think and make decisions on its own, kind of like a robot that can solve problems and learn new things!
------------------------------------------------------------
ROLE: You are a poet. Answer with a 3-line poem.
ANSWER: In silicon halls, a mind is born,
An AI agent, with logic sworn,
Intelligent being, with code forlorn.
------------------------------------------------------------
ROLE: You are an engineer. Answer in one technical sentence.
ANSWER: An AI agent is a software program that perceives its environment, takes actions, and learns from its experiences to achieve a specific goal or set of goals through a combination of machine learning, decision-making, and reasoning algorithms.
------------------------------------------------------------


## ✍️ Try it yourself

1. Change `my_question` below to anything you like and run the cell.
2. Change `my_role` to give the model a different personality.

In [8]:
my_question = "Give me three tips for learning Python."
my_role = "You are a friendly teacher. Use bullet points."

print(ask(my_question, system=my_role))

Here are three tips for learning Python:

• **Start with the basics**: Before diving into advanced topics, make sure you have a solid understanding of the fundamentals of Python. This includes learning about data types, variables, control structures, functions, and modules. Online resources like Codecademy, Python.org, and W3Schools are great places to start.

• **Practice, practice, practice**: Practice is key to learning any programming language, including Python. Try solving problems on platforms like LeetCode, HackerRank, or Project Euler. You can also work on personal projects, such as building a simple game or a web scraper. The more you practice, the more comfortable you'll become with the language.

• **Use real-world examples and projects**: Learning Python by working on real-world projects is a great way to apply your knowledge and stay motivated. Look for projects that interest you, such as building a web scraper, a chatbot, or a game. This will help you see the practical ap

## 🧰 If something goes wrong

| Message | What to do |
|---|---|
| `Ollama is not running` | In a terminal: `sudo systemctl start ollama` |
| `llama3.2 is missing` | In a terminal: `ollama pull llama3.2` |
| `ModuleNotFoundError: ollama` | Activate the environment (`source .venv/bin/activate`), run `python -m pip install -r requirements.txt`, restart the kernel |
| `externally-managed-environment` | You ran pip outside `.venv` — activate it first |
| Answers are very slow | Normal on a computer without a GPU. Try the smaller model: `ollama pull llama3.2:1b` and set `MODEL = "llama3.2:1b"` |
| Strange errors after editing cells | Menu **Kernel → Restart Kernel and Run All Cells** |

## ✅ Summary
- Ollama runs the model on **your** computer; Python talks to it with `ollama.chat()`.
- `ask(prompt, system)` is our small helper: system message = role, user message = question.
- Replies tell us the number of tokens and the time taken.

**Next:** Notebook 01 — making the model work in several steps (an *agentic workflow*).